In [ ]:
%load_ext autoreload
%autoreload 2

from tasks.diffusion import GaussianDiffusionTask
from tasks.autoencoder import AETask
import os
import torch
from tqdm import tqdm
from model.gaussian_diffusion import *
from evaluate import load
import einx
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import torch
from tasks.autoencoder import AETask
import os

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f6480be7730>>
Traceback (most recent call last):
  File "/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [2]:
ae_task = AETask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/",
                'yutrpqk5/',
                "last.ckpt",
            ),
            strict=False,
        )

Instantiating a decoder T5Attention without passing `layer_idx` is not recommended and will to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/lightning/pytorch/core/saving.py:191: Found keys that are in the model state dict but not in the checkpoint: ['decoder.prompt_generator.0.weight', 'decoder.prompt_generator.2.weight', 'decoder.prompt_generator.3.layer

In [10]:
ae_task.encoder.model.model.encoder.config.d_model

1024

In [4]:
ae_task.setup()

In [5]:
batch = next(iter(ae_task.train_dataloader()))

In [19]:
ae_task.encoder.training = False

In [20]:
z = ae_task.encoder(batch["input_ids_enc"].cuda(), batch["attention_mask_enc"].cuda())

In [21]:
# Compute cache for z
prompt = ae_task.decoder.generate_prompt(z).half()
cache = ae_task.decoder.backbone(
    inputs_embeds=prompt,
    use_cache=True,
).past_key_values
# Position of the BOS should be after the prefix (like in training)
cache_position = torch.tensor([prompt.shape[1]])

In [22]:
# Autoregressive generation from BOS token with cached z (nucleus sampling)
bos = torch.full(
    (z.shape[0], 1),
    ae_task.decoder.tokenizer.bos_token_id,
    device=z.device,
    dtype=torch.long,
)
output = ae_task.decoder.backbone.generate(
    input_ids=bos,
    past_key_values=cache,
    cache_position=cache_position,
    max_length=150,
    do_sample=True,
    top_p=0.92,
    top_k=50,
    num_beams=1,
    temperature=0.9,
    return_dict_in_generate=True,
    use_cache=True,
    pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
    eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
)

output = output.sequences[:, 1:]  # Remove BOS token

In [23]:
ae_task.decoder.tokenizer.batch_decode(output, skip_special_tokens=True)

[' the [ the [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ [ ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ] ]\n\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.\n\n.',
 '\n\n\n\n\n\n\n.\n.\n\n\n. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 ' . . the the the the you the the the the. He had the urge. He went back the store. He gave himself a lot of tips.\n\n. . .\n\n . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 ' the . She [ She had the sheet. She had the sheets. She had the

In [5]:
from functools import partial
from model.gaussian_diffusion import time_to_alpha, get_sampling_schedule
import torch

In [18]:
time_to_alpha(t=torch.tensor(0.5), alpha_schedule=get_sampling_schedule("cosine"),scale=3.0)

tensor(0.9000)

In [2]:
from transformers import T5EncoderModel, T5Tokenizer, T5Model, AutoModelForCausalLM
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import InputModule

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class PretrainedDAE(InputModule):
    def __init__(self, name):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia",
            trust_remote_code=True,
        )
        self.tokenizer = T5Tokenizer.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia"
        )
        del self.model.decoder, self.model.dec_emb, self.model.lm_head

    def tokenize(self, texts):
        return self.tokenizer.batch_encode_plus(
            texts, return_tensors="pt", padding=True, 
        )

    def save(self, path):
        pass

    def get_sentence_embedding_dimension(self):
        return self.model.bottleneck.out_proj.out_features

    def forward(self, features):
        hidden_states = self.model.encoder(**features).last_hidden_state
        attention_mask = features["attention_mask"]

        hidden_states = hidden_states.repeat(
            attention_mask.shape[0] // hidden_states.shape[0], 1, 1
        )  # during contrastive search, attn mask can have higher batch size than hidden_state
        mask_expanded = attention_mask.to(dtype=hidden_states.dtype).unsqueeze(-1).expand(hidden_states.shape)
        mean_pooled_embedding = torch.sum(
            hidden_states * mask_expanded, 1
        ) / torch.clamp(mask_expanded.sum(1), min=1e-9)
        unscaled_latent, attn_weights = self.model.bottleneck(
            mean_pooled_embedding.unsqueeze(1),
            hidden_states,
            hidden_states,
            need_weights=False,
            # torch MHA attn_mask has opposite signs to HF T5 masks... sigh
            attn_mask=attention_mask.to(dtype=hidden_states.dtype)
            .unsqueeze(1)
            .repeat_interleave(self.model.num_heads, dim=0),
        )
        latent = self.model.bottleneck_scale * F.normalize(unscaled_latent, p=2, dim=2)

        return {"sentence_embedding": latent.squeeze(1)}

In [5]:
semb = PretrainedDAE("xl")
test = SentenceTransformer(modules=[semb], model_kwargs={"torch_dtype": "float16"})

Instantiating a decoder T5Attention without passing `layer_idx` is not recommended and will to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.35it/s]


In [105]:
a = test.encode(sentences=["Allo mon nom est Léo", "Wassup my boii"], convert_to_tensor=True)

In [98]:
b = test.encode(sentences=["Allo mon nom est Léo", "Wassup my boii"], convert_to_tensor=True)

In [115]:
(a - b.to(dtype=torch.float32)).norm(dim=1)

tensor([1.4720, 1.3557], device='cuda:0')

In [117]:
a.norm(dim=1)

tensor([1.7872, 1.7872], device='cuda:0')

In [118]:
b.to(dtype=torch.float32).norm(dim=1)

tensor([1.7869, 1.7870], device='cuda:0')

In [1]:
from sentence_transformers import SentenceTransformer

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
tok = T5Tokenizer.from_pretrained("thesephist/contra-bottleneck-t5-xl-wikipedia")
model = AutoModelForCausalLM.from_pretrained(
    f"thesephist/contra-bottleneck-t5-xl-wikipedia", trust_remote_code=True
).cuda()

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.22it/s]


In [36]:
inputs = tok.batch_encode_plus(["Allo mon nom est Léo", "Wassup my boii"], return_tensors='pt', padding=True).to('cuda')
decoder_inputs = tok.batch_encode_plus(['']*2, return_tensors='pt', padding=True)
b=model(
    **inputs,
    decoder_input_ids=decoder_inputs['input_ids'],
    encode_only=True,
)

In [38]:
a -b

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0',
       grad_fn=<SubBackward0>)

In [51]:
del model.decoder

In [26]:
encoder_outputs = model.encoder(**inputs)

In [30]:
hidden_states = encoder_outputs.last_hidden_state
attention_mask = inputs['attention_mask']

In [53]:
from sentence_transformers.models import Transformer

In [ ]:
hidden_states = hidden_states.repeat(
    attention_mask.shape[0] // hidden_states.shape[0], 1, 1
)  # during contrastive search, attn mask can have higher batch size than hidden_state
mask_expanded = attention_mask.float().unsqueeze(-1).expand(hidden_states.shape)
mean_pooled_embedding = torch.sum(hidden_states * mask_expanded, 1) / torch.clamp(
    mask_expanded.sum(1), min=1e-9
)
unscaled_latent, attn_weights = model.bottleneck(
    mean_pooled_embedding.unsqueeze(1),
    hidden_states,
    hidden_states,
    need_weights=False,
    # torch MHA attn_mask has opposite signs to HF T5 masks... sigh
    attn_mask=attention_mask.float()
    .unsqueeze(1)
    .repeat_interleave(model.num_heads, dim=0),
)
latent = model.bottleneck_scale * F.normalize(unscaled_latent, p=2, dim=2)

In [68]:
class PretrainedDAE(InputModule):
    def __init__(self, name):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia", trust_remote_code=True
        )
        self.tokenizer = T5Tokenizer.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia"
        )
        del self.model.decoder, self.model.dec_emb, self.model.lm_head

    def tokenize(self, texts):
        return self.tokenizer.batch_encode_plus(texts, return_tensors="pt", padding=True)
    
    def save(self, path):
        pass
    
    def forward(self, features):
        hidden_states = self.model.encoder(**features).last_hidden_state
        attention_mask = features["attention_mask"]

        hidden_states = hidden_states.repeat(
            attention_mask.shape[0] // hidden_states.shape[0], 1, 1
        )  # during contrastive search, attn mask can have higher batch size than hidden_state
        mask_expanded = attention_mask.float().unsqueeze(-1).expand(hidden_states.shape)
        mean_pooled_embedding = torch.sum(hidden_states * mask_expanded, 1) / torch.clamp(
            mask_expanded.sum(1), min=1e-9
        )
        unscaled_latent, attn_weights = self.model.bottleneck(
            mean_pooled_embedding.unsqueeze(1),
            hidden_states,
            hidden_states,
            need_weights=False,
            # torch MHA attn_mask has opposite signs to HF T5 masks... sigh
            attn_mask=attention_mask.float()
            .unsqueeze(1)
            .repeat_interleave(self.model.num_heads, dim=0),
        )
        latent = self.model.bottleneck_scale * F.normalize(unscaled_latent, p=2, dim=2)

        return latent.squeeze(1)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.65it/s]


/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py:1080: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  embeddings = out_features[output_value]


IndexError: too many indices for tensor of dimension 2

In [3]:
from sentence_transformers import SentenceTransformer

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/sentence-t5-base', model_kwargs={"torch_dtype": "float16"})

In [13]:
batch = list(model.children())[0].tokenizer(sentences, return_tensors="pt", padding=True)

In [48]:
modules = list(model.children())

In [57]:
model.tokenizer

T5TokenizerFast(name_or_path='sentence-transformers/sentence-t5-base', vocab_size=32100, model_max_length=256, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<extra_id_42

In [54]:
out.sentence_embedding

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16, grad_fn=<DivBackward0>)

In [26]:
xd =list(model.children())[0].forward(
    batch.to('cuda')
)

In [31]:
list(model.children())[0].encode(sentences, convert_to_tensor=True)

AttributeError: 'Transformer' object has no attribute 'encode'

In [35]:
from torch.nn import Sequential

In [40]:
seq = Sequential(*list(model.children())[1:])

In [43]:
out = seq(xd)

In [46]:
out_ = model.encode(sentences, convert_to_tensor=True)

In [47]:
out_

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16)

In [45]:
out.sentence_embedding

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16, grad_fn=<DivBackward0>)

In [30]:
res = xd.token_embeddings

In [ ]:
embeddings = model.encode(sentences, convert_to_tensor=True)

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16)

In [1]:
from transformers import T5EncoderModel, T5Tokenizer, T5Model
from sentence_transformers import SentenceTransformer
# import AutoModel
from transformers import AutoModel, AutoTokenizer

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer(
    "sentence-transformers/sentence-t5-xl",
    device="cuda",
    model_kwargs={"torch_dtype": "float16"},
)

In [7]:
model.get_sentence_embedding_dimension()

768

In [6]:
list(model.children())

[Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'T5EncoderModel'}),
 Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True}),
 Dense({'in_features': 1024, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'}),
 Normalize()]

In [8]:
input_ids= tok("Hello, my dog is cute", return_tensors="pt").input_ids
xd = model(input_ids)

In [9]:
xd.last_hidden_state.shape

torch.Size([1, 7, 1024])

In [2]:
os.environ["LATENT_CONTROL_CKPT_DIR"] = (
    "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints"
)

In [3]:
ae_task = AETask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/cm9ujm08/"
                "last.ckpt",
            ),
            strict=False
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.


In [4]:
ae_task = ae_task.cuda()

In [5]:
ae_task.setup()

In [6]:
batch = next(iter(ae_task.val_dataloader()))

In [7]:
with torch.no_grad():
    z = ae_task.encoder(
        input_ids=batch["input_ids_enc"].cuda(), 
        attention_mask=batch["attention_mask_enc"].cuda()
    )

In [8]:
z = z[:10]

In [21]:
position_ids = torch.zeros(
    z.shape[0], z.shape[1] + 1, device=z.device, dtype=torch.long
)
bos_emb = ae_task.decoder.backbone.get_input_embeddings()(
    torch.tensor(ae_task.decoder.tokenizer.bos_token_id).cuda()
)
bos_emb = einx.rearrange("d -> b 1 d", bos_emb, b=z.shape[0])
z_with_bos = torch.cat([z, bos_emb], dim=1).half()

In [25]:
with torch.no_grad():
    output = ae_task.decoder.backbone.generate(
        inputs_embeds=z_with_bos,
        #position_ids=position_ids,
        max_length=200,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        num_beams=1,
        temperature=0.9,
        return_dict_in_generate=True,
        use_cache=True,
        pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
        eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
    )

In [26]:
seqs = ae_task.decoder.tokenizer.batch_decode(output.sequences, skip_special_tokens=True)

In [27]:
seqs

[' They They had to to to to to to to. to. When had called me and she called her. She said thank you. She said goodbye in housing.',
 ' my my taught me when when. I has. My teacher began to recently. This continues to. This is.',
 ' Tom took a a a a         ',
 ' Martin heard heard a he he he he he he he he he he he he he he had seen a man staring at mask at clown..',
 ' he he he he t he . . . t . t . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . a . . a . . . a . a . a . a . a 1 . a . a C TOTTERLOTTERLOTTERLOTTERLOTTERLOTTERLOTTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTATERL TOM L T L TAKE TOM L T TATERL T TA MAN TTA TE TOM TOM TOM TOM L T T TTA',
 "Thethe'whenthe the when the to when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when whe

In [50]:
prefill = ae_task.decoder.backbone(
    inputs_embeds=z.half(),
    use_cache=True,
)
cache = prefill.past_key_values
cache_position = torch.tensor([0])
bos = torch.full(
    (z.shape[0], 1),
    ae_task.decoder.tokenizer.bos_token_id,
    device=z.device,
    dtype=torch.long,
)

In [51]:
with torch.no_grad():
    output = ae_task.decoder.backbone.generate(
        input_ids=bos,
        past_key_values=cache,
        cache_position=cache_position,
        max_length=200,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        num_beams=1,
        temperature=0.9,
        return_dict_in_generate=True,
        use_cache=True,
        pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
        eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
        )

In [49]:
seqs = ae_task.decoder.tokenizer.batch_decode(output.sequences, skip_special_tokens=True)
seqs

[' they they had to to to to to to to. They And When He There She You She I I I C My advice said said housing was was was. This was was was.',
 ' My grandmother taught me when. I my had. My grandmother had done this projects. I continue to projects. This created.',
 ' Tom took a a a          ',
 ' Martin heard heard a he he he he he he he he he he he he he had a look out of clown face in clown face..',
 ' he he he he          in K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K',
 'TheWhen the the the when the to the the to to to to to.. to. to. to. to, they with ( 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 

In [9]:
diffusion_task = GaussianDiffusionTask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/a1pq0e97/"
                "last.ckpt",
            ),
            strict=False,
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [10]:
diffusion_task = diffusion_task.cuda()

In [11]:
diffusion_task.setup()

In [12]:
mauve = diffusion_task.get_mauve_score()

Featurizing q:  88%|████████▊ | 7/8 [00:04<00:00,  1.67it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 214.00 MiB. GPU 0 has a total capacity of 44.64 GiB of which 118.44 MiB is free. Including non-PyTorch memory, this process has 44.52 GiB memory in use. Of the allocated memory 39.63 GiB is allocated by PyTorch, and 4.36 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [7]:
mauve

0.11225014485875065

In [1]:
from transformers.models.auto.modeling_auto import AutoModelForCausalLM
from transformers.models.auto.tokenization_auto import AutoTokenizer


tok = AutoTokenizer.from_pretrained('thesephist/contra-bottleneck-t5-large-wikipedia', model_max_length=512)
model = AutoModelForCausalLM.from_pretrained('thesephist/contra-bottleneck-t5-large-wikipedia', trust_remote_code=True)

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A new version of the following files was downloaded from https://huggingface.co/thesephist/contra-bottleneck-t5-large-wikipedia:
- bottleneck_t5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Instantiating a decoder T5Attention without passing `layer_idx` is not recommended and will to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


In [8]:
list(model.modules())[3]

ModuleList(
  (0): T5Block(
    (layer): ModuleList(
      (0): T5LayerSelfAttention(
        (SelfAttention): T5Attention(
          (q): Linear(in_features=1024, out_features=1024, bias=False)
          (k): Linear(in_features=1024, out_features=1024, bias=False)
          (v): Linear(in_features=1024, out_features=1024, bias=False)
          (o): Linear(in_features=1024, out_features=1024, bias=False)
          (relative_attention_bias): Embedding(32, 16)
        )
        (layer_norm): T5LayerNorm()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (1): T5LayerFF(
        (DenseReluDense): T5DenseGatedActDense(
          (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
          (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
          (wo): Linear(in_features=2816, out_features=1024, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
          (act): NewGELUActivation()
        )
        (layer_norm): T5LayerNorm()
        (d

In [79]:
import wandb
from typing import Any, List, Optional, Dict

def get_runs_by_config(
    entity: str,
    project: str,
    config_filters: Dict[str, Any],
    state: Optional[str] = None
) -> List[str]:
    """
    Retrieve run IDs from a W&B project that match specific config values.
    
    Args:
        entity: W&B entity name
        project: W&B project name
        config_filters: Dictionary of config key-value pairs to filter by
                       e.g., {"sweep_id": "dae_sweep", "task.diffusion.model_type": "transformer"}
        state: Optional run state filter ("finished", "running", "crashed", etc.)
    
    Returns:
        List of run IDs that match the criteria
    """
    api = wandb.Api()
    
    # Build the filter dictionary for the API
    filters = {}
    if state:
        filters["state"] = state
    
    # Get all runs first, then filter manually since W&B's config filtering can be unreliable
    runs = api.runs(f"{entity}/{project}", filters=filters)
    
    matching_run_ids = []
    
    for run in runs:
        # Check if all config filters match
        matches_all = True
        for config_key, expected_value in config_filters.items():
            # Navigate nested config using dot notation
            config_value = run.config
            for key_part in config_key.split('.'):
                if isinstance(config_value, dict) and key_part in config_value:
                    config_value = config_value[key_part]
                else:
                    config_value = None
                    break
            
            if config_value != expected_value:
                matches_all = False
                break
        
        if matches_all:
            matching_run_ids.append(run.id)
    
    return matching_run_ids

In [82]:
def list_run_ids(
    entity: str,
    project: str,
    where: Optional[Dict[str, Any]] = None,
    state: Optional[str] = None,
) -> List[str]:
    api = wandb.Api()
    filters: Dict[str, Any] = {}

    # Convert dot-paths to "config.<dotpath>" for W&B filters
    if where:
        for k, v in where.items():
            filters[f"config.{k}"] = v

    if state:
        filters["state"] = state  # e.g., "finished", "failed", "running", etc.

    runs = api.runs(f"{entity}/{project}", filters=filters)
    return [r.id for r in runs]

In [91]:
api = wandb.Api()

In [ ]:
filters = {
    "config.sweep_id": "dae_sweep",
}
runs = api.runs(f"guillaume-lajoie/sentence_diffusion", filters=filters)
ids = [run.id for run in runs]

In [98]:
ae_task.decoder.tokenizer.bos_token_id, ae_task.decoder.tokenizer.eos_token_id, ae_task.decoder.tokenizer.pad_token_id

(50256, 50256, 50257)

In [ ]:
list_run_ids(
    entity="guillaume-lajoie",
    project="sentence_diffusion",
    filters={
        "config.sweep_id": "dae_sweep",
    },
    state="finished"
)

[]

In [87]:
run_ids = get_runs_by_config(
    entity="guillaume-lajoie",
    project="sentence_diffusion",
    config_filters={
        "sweep_id": {"$in": ["dae_sweep"]},  # Has diffusion task
    },
    state="finished"
)

In [88]:
run_ids

[]